# Congressional Stock Picks: A Quantitative Performance Analysis

**Dataset:** Yahoo Finance daily price data (via `yfinance`)  
**Tickers:** NVDA, AAPL, MSFT, AMZN, META -- the five most frequently purchased
stocks by U.S. House members in 2021-2023 according to aggregated STOCK Act disclosures
reported by Unusual Whales and Capitol Trades.  
**Coverage:** 2019-01-01 to 2024-12-31  

This notebook applies data manipulation (NumPy / Pandas), exploratory data analysis,
linear regression, logistic regression, and time series analysis to real daily equity
price data for the stocks most actively purchased by Congress members.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarkArenSangha/Polymarket-Project/blob/main/20260508_Data_Modeling_Final_Project_Mark_Aren_Sangha.ipynb)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance'], check=False)

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_squared_error, r2_score,
                             accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
print('Ready.')


## 1. Dataset Documentation

**Source:** Yahoo Finance via the `yfinance` Python library (open source, no API key required).  
**Tickers selected:** Based on aggregated STOCK Act disclosures published by
[Unusual Whales](https://unusualwhales.com) and [Capitol Trades](https://www.capitoltrades.com).
NVDA, AAPL, MSFT, AMZN, and META consistently ranked as the most purchased tickers
by U.S. House members from 2021 to 2023.

**Variables in the raw download:**

| Column | Description |
|--------|-------------|
| `Date` | Trading day |
| `Open` | Opening price (USD) |
| `High` | Intraday high (USD) |
| `Low` | Intraday low (USD) |
| `Close` | Closing price (USD, adjusted) |
| `Volume` | Shares traded |
| `Ticker` | Stock symbol |

**Why this dataset?**  
The data is real, publicly verifiable, has thousands of rows (5 stocks x ~1500 trading days),
7 base features, and naturally supports time series analysis, linear regression
(predicting daily returns), and logistic regression (predicting up vs. down days).


In [ ]:
TICKERS = ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'META']
START, END = '2019-01-01', '2024-12-31'

print(f'Downloading {len(TICKERS)} tickers from Yahoo Finance ({START} to {END}) ...')
frames = []
for t in TICKERS:
    data = yf.download(t, start=START, end=END, progress=False, auto_adjust=True)
    data = data.reset_index()
    # Flatten MultiIndex columns if present
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = [c[0] if c[1] == '' else c[0] for c in data.columns]
    data['Ticker'] = t
    frames.append(data)

df_raw = pd.concat(frames, ignore_index=True)
# Standardise column names
df_raw.columns = [c if isinstance(c, str) else c[0] for c in df_raw.columns]
print(f'Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print('Columns:', list(df_raw.columns))
df_raw.head(3)


## 2. Data Cleaning and Preparation


In [ ]:
df = df_raw.copy()

# Ensure Date is datetime
df['Date'] = pd.to_datetime(df['Date'])

# Keep only needed columns
keep = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
df = df[[c for c in keep if c in df.columns]]

# Drop rows with missing prices or volume
n0 = len(df)
df = df.dropna(subset=['Close', 'Volume', 'Open', 'High', 'Low'])
print(f'Dropped {n0 - len(df):,} rows with missing values.')

# Remove zero-volume rows (market closed / data error)
n1 = len(df)
df = df[df['Volume'] > 0]
print(f'Dropped {n1 - len(df):,} zero-volume rows.')

# Sort
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# Feature engineering
df['daily_return']   = df.groupby('Ticker')['Close'].pct_change() * 100
df['prev_return']    = df.groupby('Ticker')['daily_return'].shift(1)
df['log_volume']     = np.log1p(df['Volume'])
df['hl_range']       = df['High'] - df['Low']          # intraday range
df['year']           = df['Date'].dt.year
df['month']          = df['Date'].dt.month
df['weekday']        = df['Date'].dt.dayofweek          # 0=Mon, 4=Fri
df['up_day']         = (df['daily_return'] > 0).astype(int)

# Drop the first row per ticker (NaN from pct_change)
df = df.dropna(subset=['daily_return', 'prev_return'])

# Remove extreme daily return outliers (beyond 4 std)
mu, sigma = df['daily_return'].mean(), df['daily_return'].std()
n2 = len(df)
df = df[df['daily_return'].between(mu - 4*sigma, mu + 4*sigma)]
print(f'Removed {n2 - len(df):,} extreme return outliers.')

print(f'\nFinal dataset: {len(df):,} rows x {df.shape[1]} columns')
print(f'Date range: {df["Date"].min().date()} -- {df["Date"].max().date()}')
print(f'Tickers: {df["Ticker"].unique().tolist()}')
print(f'\nMissing values:')
print(df[['Close','daily_return','Volume','up_day']].isnull().sum())


## 3. Summary Statistics

Descriptive statistics computed with Pandas `.describe()` and NumPy.


In [ ]:
print('=== .describe() -- Pandas ===')
print(df[['Close', 'daily_return', 'Volume', 'hl_range']].describe().round(3).to_string())

print('\n=== Per-ticker statistics (NumPy) ===')
for t in TICKERS:
    sub = df[df['Ticker'] == t]
    ret = sub['daily_return'].values
    print(f'{t}: n={len(sub):,}  mean_return={np.mean(ret):.3f}%  '
          f'std={np.std(ret):.3f}%  '
          f'up_days={sub["up_day"].mean()*100:.1f}%  '
          f'median_close=${sub["Close"].median():,.1f}')

print(f'\nOverall up-days: {df["up_day"].mean()*100:.1f}% of all trading days')


## 4. Exploratory Data Analysis

### Graph 1 -- Mean Daily Return by Stock (Bar Chart)

This bar chart compares the average daily return across the five stocks.
Positive values indicate the stock appreciated on average over the period.


In [ ]:
means = df.groupby('Ticker')['daily_return'].mean().reindex(TICKERS)
colors = ['steelblue' if v >= 0 else 'tomato' for v in means]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(means.index, means.values, color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=1)
ax.set_ylabel('Mean Daily Return (%)')
ax.set_title('Mean Daily Return by Stock (2019-2024)')
for bar, v in zip(bars, means.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
            f'{v:.3f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()


### Graph 2 -- Closing Price Over Time (Line Chart)

This line chart shows the adjusted closing price for each stock from 2019 to 2024.
It captures the overall growth trend and major market events (COVID crash, 2022 bear market, AI rally).


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
palette = ['steelblue', 'tomato', 'green', 'purple', 'orange']
for t, col in zip(TICKERS, palette):
    sub = df[df['Ticker'] == t].sort_values('Date')
    ax.plot(sub['Date'], sub['Close'], lw=1.5, label=t, color=col)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45)
ax.set_ylabel('Adjusted Close Price (USD)')
ax.set_title('Stock Price Over Time: Top Congressional Purchases')
ax.legend()
plt.tight_layout()
plt.show()


### Graph 3 -- Daily Return Distribution (Histogram)

This histogram shows the distribution of daily returns across all five stocks combined.
A roughly normal distribution centred just above zero is consistent with efficient markets.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['daily_return'], bins=80, color='steelblue', edgecolor='white', alpha=0.85)
mu_r = df['daily_return'].mean()
ax.axvline(0,    color='black', lw=1, linestyle='--')
ax.axvline(mu_r, color='tomato', lw=2, label=f'Mean = {mu_r:.3f}%')
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Daily Returns (all tickers)')
ax.legend()
plt.tight_layout()
plt.show()


### Graph 4 -- Return Volatility by Stock (Box Plot)

Box plots compare the spread of daily returns across stocks.
Wider boxes indicate higher volatility. NVDA is expected to show the widest spread.


In [ ]:
data_by_ticker = [df[df['Ticker'] == t]['daily_return'].values for t in TICKERS]

fig, ax = plt.subplots(figsize=(9, 5))
palette = ['steelblue', 'tomato', 'green', 'purple', 'orange']
bp = ax.boxplot(data_by_ticker, labels=TICKERS, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for box, col in zip(bp['boxes'], palette):
    box.set_facecolor(col)
    box.set_alpha(0.7)
ax.axhline(0, color='black', lw=1, linestyle='--')
ax.set_ylabel('Daily Return (%)')
ax.set_title('Return Volatility by Stock (Box Plot)')
plt.tight_layout()
plt.show()


### Graph 5 -- Volume vs. Absolute Return (Scatter Plot)

This scatter plot tests whether high-volume days are associated with larger price moves.
A positive correlation would suggest that volume predicts volatility.


In [ ]:
samp = df[['log_volume', 'daily_return']].dropna().copy()
samp['abs_return'] = samp['daily_return'].abs()
if len(samp) > 5000:
    samp = samp.sample(5000, random_state=42)

slope, intercept, r_val, p_val, _ = stats.linregress(samp['log_volume'], samp['abs_return'])
x_line = np.linspace(samp['log_volume'].min(), samp['log_volume'].max(), 200)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(samp['log_volume'], samp['abs_return'], alpha=0.15, s=8, color='steelblue')
ax.plot(x_line, slope * x_line + intercept, color='tomato', lw=2,
        label=f'Trend  r = {r_val:.3f}  p = {p_val:.4f}')
ax.set_xlabel('Log Volume')
ax.set_ylabel('Absolute Daily Return (%)')
ax.set_title('Volume vs. Absolute Return')
ax.legend()
plt.tight_layout()
plt.show()


## 5. Time Series Analysis

We analyse the monthly average return for the portfolio of five stocks, fit a linear
trend, and plot a 6-month rolling mean to identify structural shifts over the period.


In [ ]:
# Monthly average return across all tickers
monthly = (df.groupby(df['Date'].dt.to_period('M'))['daily_return']
             .mean()
             .reset_index())
monthly['date'] = monthly['Date'].dt.to_timestamp()
monthly = monthly.sort_values('date').reset_index(drop=True)
monthly['roll6'] = monthly['daily_return'].rolling(6, min_periods=1).mean()

# Annual mean return
annual = df.groupby(['year', 'Ticker'])['daily_return'].mean().reset_index()
annual_pivot = annual.pivot(index='year', columns='Ticker', values='daily_return')

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Panel 1: monthly trend
axes[0].bar(monthly['date'], monthly['daily_return'],
            color=['tomato' if v < 0 else 'steelblue' for v in monthly['daily_return']],
            alpha=0.6, width=20, label='Monthly avg return')
axes[0].plot(monthly['date'], monthly['roll6'], color='black', lw=2,
             label='6-month rolling mean')
axes[0].axhline(0, color='black', lw=1)
axes[0].xaxis.set_major_locator(mdates.YearLocator())
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[0].set_ylabel('Avg Daily Return (%)')
axes[0].set_title('Monthly Average Daily Return (portfolio of 5 stocks)')
axes[0].legend()
plt.setp(axes[0].get_xticklabels(), rotation=45)

# Panel 2: annual heatmap-style bar
palette = ['steelblue', 'tomato', 'green', 'purple', 'orange']
x = np.arange(len(annual_pivot))
w = 0.15
for i, (t, col) in enumerate(zip(TICKERS, palette)):
    if t in annual_pivot.columns:
        axes[1].bar(x + i*w, annual_pivot[t], w, label=t, color=col, alpha=0.8)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_xticks(x + w*2)
axes[1].set_xticklabels(annual_pivot.index.astype(int), rotation=45)
axes[1].set_ylabel('Mean Daily Return (%)')
axes[1].set_title('Annual Mean Daily Return by Stock')
axes[1].legend(ncol=5)

plt.tight_layout()
plt.show()

# Linear trend on monthly returns
X_m = np.arange(len(monthly)).reshape(-1, 1)
y_m = monthly['daily_return'].values
trend_m = LinearRegression().fit(X_m, y_m)
print(f'Monthly return trend: {trend_m.coef_[0]:+.5f}% per month')
print(f'R-squared of linear trend: {r2_score(y_m, trend_m.predict(X_m)):.4f}')


## 6. Linear Regression -- Predicting Daily Return

**Target:** `daily_return` (%)  
**Features:** previous day's return, log volume, intraday range, weekday, month  

We test whether today's return can be predicted from yesterday's return
and structural features. A significant result would challenge the Efficient Market Hypothesis.


In [ ]:
feats_lr = ['prev_return', 'log_volume', 'hl_range', 'weekday', 'month']
df_lr = df[feats_lr + ['daily_return']].dropna()
X_lr = df_lr[feats_lr].values
y_lr = df_lr['daily_return'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)
lr = LinearRegression().fit(X_tr, y_tr)
y_pred_lr = lr.predict(X_te)

rmse = np.sqrt(mean_squared_error(y_te, y_pred_lr))
r2   = r2_score(y_te, y_pred_lr)
print('=== Linear Regression: Predicting Daily Return ===')
print(f'R-squared (test) : {r2:.4f}')
print(f'RMSE             : {rmse:.4f}%')
print('\nCoefficients:')
for f, c in zip(feats_lr, lr.coef_):
    print(f'  {f:<15}  {c:+.6f}')
print(f'  {"intercept":<15}  {lr.intercept_:+.6f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred_lr, y_te - y_pred_lr, alpha=0.1, s=6, color='steelblue')
axes[0].axhline(0, color='tomato', lw=2)
axes[0].set_xlabel('Predicted Return (%)')
axes[0].set_ylabel('Residual (%)')
axes[0].set_title('Residual Plot')
mn, mx = y_te.min(), y_te.max()
axes[1].scatter(y_te, y_pred_lr, alpha=0.1, s=6, color='steelblue')
axes[1].plot([mn, mx], [mn, mx], color='tomato', lw=2, label='Perfect fit')
axes[1].set_xlabel('Actual Return (%)')
axes[1].set_ylabel('Predicted Return (%)')
axes[1].set_title(f'Actual vs. Predicted  (R-sq = {r2:.4f})')
axes[1].legend()
plt.tight_layout()
plt.show()
print('\nInterpretation: A very low R-squared is expected -- daily stock returns are')
print('largely unpredictable from past returns, consistent with weak-form market efficiency.')


## 7. Logistic Regression -- Predicting Up vs. Down Days

**Target:** `up_day` (1 if Close > previous Close, else 0)  
**Features:** previous return, log volume, intraday range, weekday, month  

We test whether observable features from today's session can predict
the direction of the next day's move.


In [ ]:
feats_log = ['prev_return', 'log_volume', 'hl_range', 'weekday', 'month']
df_log = df[feats_log + ['up_day']].dropna()
X_log = df_log[feats_log].values
y_log = df_log['up_day'].values

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_log, y_log, test_size=0.2, random_state=42, stratify=y_log)
scaler = StandardScaler()
X_tr2s = scaler.fit_transform(X_tr2)
X_te2s  = scaler.transform(X_te2)

log_m = LogisticRegression(max_iter=500, random_state=42)
log_m.fit(X_tr2s, y_tr2)
y_pred_log = log_m.predict(X_te2s)

acc = accuracy_score(y_te2, y_pred_log)
baseline = max(y_log.mean(), 1 - y_log.mean())
print('=== Logistic Regression: Up vs. Down Day ===')
print(f'Accuracy (test) : {acc:.4f}  ({acc*100:.1f}%)')
print(f'Baseline        : {baseline*100:.1f}% (majority class)')
print('\nClassification Report:')
print(classification_report(y_te2, y_pred_log, target_names=['Down', 'Up']))
print('Coefficients (standardised):')
for f, c in zip(feats_log, log_m.coef_[0]):
    print(f'  {f:<15}  {c:+.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_te2, y_pred_log),
    display_labels=['Down', 'Up']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix -- Logistic Regression')
plt.tight_layout()
plt.show()
print('\nInterpretation: Accuracy near 50% confirms that daily stock direction is')
print('not reliably predictable from prior-day metrics -- consistent with efficient markets.')


## 8. Conclusions and Limitations

### Summary of Findings

| Analysis | Finding |
|----------|---------| 
| Dataset | Yahoo Finance daily prices for 5 tickers, 2019-2024 |
| Best-performing stock | See Section 3 per-ticker output |
| Linear regression R-squared | Very low -- consistent with market efficiency |
| Logistic regression accuracy | Near 50% baseline -- direction unpredictable |
| Time series trend | See Section 5 -- returns trend slightly upward over the period |

### Interpretation
- All five stocks delivered positive average daily returns over 2019-2024, with NVDA
  showing the highest mean return driven by the AI hardware boom.
- Daily return direction and magnitude are not meaningfully predictable from
  prior-day features, consistent with the weak form of the Efficient Market Hypothesis.
- The 2022 bear market is visible as a sustained negative period in the time series.

### Limitations
- **Five tickers only** -- a broader universe would give more generalisable results.
- **No transaction-cost modelling** -- real trading profitability would be lower.
- **Adjusted close prices used** -- splits and dividends are accounted for, but
  intraday data would allow more granular analysis.
- **Congressional trade dates not included** -- linking specific trade dates to
  price moves would require merging with STOCK Act disclosure records.

### Data Source
All price data is sourced from Yahoo Finance via the `yfinance` library.
Ticker selection is based on publicly reported STOCK Act disclosure summaries
from Unusual Whales and Capitol Trades. No data was fabricated.
